# Twitter Sentiment Analysis

In [97]:
# load libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [98]:
# load data
train_df = pd.read_csv('twitter_data/train.csv')
test_df = pd.read_csv('twitter_data/test.csv')
train_df.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


In [99]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31962 entries, 0 to 31961
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      31962 non-null  int64 
 1   label   31962 non-null  int64 
 2   tweet   31962 non-null  object
dtypes: int64(2), object(1)
memory usage: 749.2+ KB


In [100]:
print('Train df duplicates :',train_df.duplicated().sum(),'\n missing: \n',train_df.isna().sum())
print('Test df duplicates : ',test_df.duplicated().sum(),'\n missing: \n',test_df.isna().sum())


Train df duplicates : 0 
 missing: 
 id       0
label    0
tweet    0
dtype: int64
Test df duplicates :  0 
 missing: 
 id       0
tweet    0
dtype: int64


In [101]:
print(f'{train_df.shape[0]} rows and {train_df.shape[1]} columns')

31962 rows and 3 columns


In [102]:
# value counts
train_df['label'].value_counts()

label
0    29720
1     2242
Name: count, dtype: int64

In [103]:
# Plot pie chart of label distribution
import plotly.express as px
fig = px.pie(train_df['label'].value_counts().reset_index(name='count').rename(columns={'index': 'label'}),
             values='count', names='label', title='Tweet Label Distribution',)
fig.show()

## 1. Lowercase

In [104]:
# lower case
train_df['tweet'] = train_df['tweet'].str.lower()
test_df['tweet'] = test_df['tweet'].str.lower()
train_df.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


## 2. Remove Html tags

In [105]:
# Remove Html tags
import re
def remove_html_tags(text):
    html_pattern = re.compile('<.*?>')
    return html_pattern.sub(r'', text)
train_df['tweet'] = train_df['tweet'].apply(lambda text: remove_html_tags(text))
train_df.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


## 3. Remove Punctuations

In [106]:
import string
exclude = string.punctuation
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [107]:
def remove_punctuations(text):
    return text.translate(str.maketrans('', '', exclude))
train_df['tweet'] = train_df['tweet'].apply(lambda text: remove_punctuations(text))
train_df.head()

,id,label,tweet
0,1,0,user when a father is dysfunctional and is so...
1,2,0,user user thanks for lyft credit i cant use ca...
2,3,0,bihday your majesty
3,4,0,model i love u take with u all the time in u...
4,5,0,factsguide society now motivation


## 4. Spelling Correction

In [108]:
from symspellpy import SymSpell, Verbosity

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym_spell.load_dictionary('frequency_dictionary_en_82_765.txt', 0, 1)

def correct_text(text):
    suggestions = sym_spell.lookup_compound(text, max_edit_distance=2)
    return suggestions[0].term if suggestions else text

# train_df['tweet'] = train_df['tweet'].apply(correct_text)

train_df.head()

,id,label,tweet
0,1,0,user when a father is dysfunctional and is so...
1,2,0,user user thanks for lyft credit i cant use ca...
2,3,0,bihday your majesty
3,4,0,model i love u take with u all the time in u...
4,5,0,factsguide society now motivation


## 5. Remove Stopwords

In [75]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])
train_df['tweet'] = train_df['tweet'].apply(lambda text: remove_stopwords(text))
train_df.head()

,id,label,tweet
0,1,0,user father dysfunctional selfish drags kids d...
1,2,0,user user thanks lyft credit cant use cause do...
2,3,0,bihday majesty
3,4,0,model love u take u time urð± ðððð...
4,5,0,factsguide society motivation


## 6. Tokenization

In [110]:
from nltk.tokenize import word_tokenize, sent_tokenize
train_df['tweet'] = train_df['tweet'].apply(word_tokenize)
train_df.head()

,id,label,tweet
0,1,0,"[user, when, a, father, is, dysfunctional, and..."
1,2,0,"[user, user, thanks, for, lyft, credit, i, can..."
2,3,0,"[bihday, your, majesty]"
3,4,0,"[model, i, love, u, take, with, u, all, the, t..."
4,5,0,"[factsguide, society, now, motivation]"


## 7. Lemmatization

In [118]:
import re
import nltk
import swifter
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag


# Download required NLTK data (only first time)
# nltk.download('punkt')
# nltk.download('wordnet')
# nltk.download('omw-1.4')
# nltk.download('averaged_perceptron_tagger')

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(word):
    """Map POS tag to first character lemmatize() accepts"""
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {'J': wordnet.ADJ, 'N': wordnet.NOUN, 'V': wordnet.VERB, 'R': wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

def clean_and_lemmatize(text):
    # Lowercase
    text = str(text).lower()
    
    # Remove mentions, URLs, hashtags, digits, punctuation, emojis
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenize and lemmatize
    words = word_tokenize(text)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in words]
    
    return ' '.join(lemmatized)

# Apply fast using swifter
train_df['clean_tweet'] = train_df['tweet'].swifter.apply(clean_and_lemmatize)

# Preview result
train_df.head()


Pandas Apply: 100%|██████████| 31962/31962 [00:19<00:00, 1617.94it/s]


,id,label,tweet,clean_tweet
0,1,0,user when a father be dysfunctional and be so ...,user when a father be dysfunctional and be so ...
1,2,0,user user thanks for lyft credit i cant use ca...,user user thanks for lyft credit i cant use ca...
2,3,0,bihday your majesty,bihday your majesty
3,4,0,model i love u take with u all the time in urð...,model i love u take with u all the time in ur
4,5,0,factsguide society now motivation,factsguide society now motivation


In [ ]:
# train_df[['label', 'clean_tweet']].to_csv('cleaned_tweets.csv', index=False)